[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blhuillier/2026B_Galactic_Dynamics/blob/main/notebooks/chapter_06/nb_06_spherical_mass_modelling.ipynb)

# Chapter 6 - Spherical Distribution Functions and Mass Modelling

**Galactic Dynamics · Sejong University · 2026B**

This notebook accompanies the numerical exercises of Chapter 6:

| Part | Exercise | Topic |
|------|----------|-------|
| A | 6.10 | Eddington inversion and the limits of anisotropy |
| B | 6.11 | The mass-anisotropy degeneracy |

Part A turns Eddington's formula (Property 6.2) into a few lines of code and uses it to find how
radially anisotropic an Osipkov-Merritt model can be before its distribution function turns negative.
Part B solves the spherical Jeans equation, projects the result on the sky, and builds two very
different models with the same line-of-sight dispersion profile, as in Figure 6.3.

Units: $G = 1$ throughout, and the scale radius of each model is 1.

Cells marked **TODO** are yours to complete. Everything else is given.

> **Running on Google Colab?** This notebook uses the `galdyn` toolkit from the
> course repository. Run the cell below first - it clones the repo and adds it
> to your Python path. On a local install (via `uv`), this cell does nothing.

In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("2026B_Galactic_Dynamics"):
        !git clone -q https://github.com/blhuillier/2026B_Galactic_Dynamics.git
    sys.path.insert(0, "2026B_Galactic_Dynamics/src")
    %pip install -q astropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq, minimize, minimize_scalar

from galdyn.plotting import set_latex_style

set_latex_style(use_tex=False)

---
## Part A - Exercise 6.10: Eddington inversion

For an isotropic model with $f = 0$ for $\mathcal E \le 0$ and $\mathrm d\rho/\mathrm d\Psi \to 0$ at
$\Psi = 0$, Eddington's formula reads

$$ f(\mathcal E) = \frac{1}{\sqrt8\,\pi^2} \int_0^{\mathcal E}
\frac{\mathrm d^2\rho}{\mathrm d\Psi^2}\, \frac{\mathrm d\Psi}{\sqrt{\mathcal E - \Psi}} . $$

The integrand is singular at $\Psi = \mathcal E$. The substitution $\mathcal E - \Psi = \mathcal E t^2$
removes the singularity: $\mathrm d\Psi/\sqrt{\mathcal E - \Psi} = 2\sqrt{\mathcal E}\,\mathrm dt$,
with $t$ running from 0 to 1.

**TODO A.1** - Implement `eddington` and test it on the Plummer model, for which
$\rho = (3/4\pi)\Psi^5$ and $f = (24\sqrt2/7\pi^3)\,\mathcal E^{7/2}$ (Example 6.1).

In [ ]:
def eddington(d2rho, E):
    # Isotropic distribution function f(E) from Eddington's formula.
    #   d2rho : callable, d^2 rho / d Psi^2 as a function of Psi
    #   E     : relative energy, 0 < E <= Psi(0)
    # TODO A.1: substitute Psi = E (1 - t^2) and integrate over t in [0, 1] with quad.
    raise NotImplementedError("TODO A.1")


def plummer_d2rho(psi):
    return 3.0 / (4.0 * np.pi) * 20.0 * psi**3


def plummer_f(E):
    return 24.0 * np.sqrt(2.0) / (7.0 * np.pi**3) * E**3.5


E_test = np.array([0.1, 0.5, 0.9])
print([eddington(plummer_d2rho, E) / plummer_f(E) for E in E_test])

**TODO A.2** - The Hernquist model has $\rho = Ma/[2\pi r(r+a)^3]$ and $\Phi = -GM/(r+a)$.
Show that, with $G = M = a = 1$, $\rho(\Psi) = \Psi^4/[2\pi(1-\Psi)]$, and write its second derivative.
*Hint:* $\Psi^4/(1-\Psi) = 1/(1-\Psi) - (1 + \Psi + \Psi^2 + \Psi^3)$.

Then compute $f(\mathcal E)$ on $0 < \mathcal E < 1$ and plot it against $1 - \mathcal E$ on logarithmic
axes. What is the slope as $\mathcal E \to 1$? Why does $f$ diverge?

In [ ]:
def hernquist_d2rho(psi):
    # TODO A.2: d^2 rho / d Psi^2 for rho = Psi^4 / [2 pi (1 - Psi)].
    raise NotImplementedError("TODO A.2")


# TODO A.2: evaluate f on E = 1 - np.logspace(-4, -0.01, 80) and plot f against 1 - E (loglog).
# Overlay a line of slope -5/2 for comparison.

**TODO A.3** - The Osipkov-Merritt version replaces $\rho$ by $\rho_Q = (1 + r^2/r_a^2)\,\rho$ and
$\mathcal E$ by $Q$ (Section 6.4). For the Hernquist model $r = 1/\Psi - 1$, so

$$ \rho_Q(\Psi) = \frac{1}{2\pi}\left[\frac{\Psi^4}{1-\Psi} + \frac{\Psi^2(1-\Psi)}{r_a^2}\right]. $$

Write `hernquist_om_d2rho(ra)`, which returns the function $\mathrm d^2\rho_Q/\mathrm d\Psi^2$.
Plot $f(Q)$ for a few values of $r_a$, then find the smallest $r_a$ for which $f(Q) \ge 0$ for all
$0 < Q < 1$, to two decimals. Where does $f$ first become negative?

In [ ]:
def hernquist_om_d2rho(ra):
    # TODO A.3: return a function of psi giving d^2 rho_Q / d Psi^2.
    raise NotImplementedError("TODO A.3")


# TODO A.3: plot f(Q) for ra in [1.0, 0.5, 0.3, 0.2].
# Then define fmin(ra) = minimum of f(Q) over Q (minimize_scalar with bounds, or a fine grid)
# and solve fmin(ra) = 0 with brentq.

**TODO A.4** - Exercise 6.3 shows that the Plummer model allows $r_a \ge 3b/4$. Express both bounds in
units of the half-mass radius ($1.305\,b$ for Plummer, $(1+\sqrt2)\,a$ for Hernquist) and explain the
difference. *Hint: Exercise 6.4.*

*Your answer here.*

---
## Part B - Exercise 6.11: the mass-anisotropy degeneracy

A massless Plummer tracer, $\nu_\star \propto (1 + r^2)^{-5/2}$, orbits in a cored halo with
$v_{\rm c}^2 = v_0^2 r^2/(r^2 + r_{\rm c}^2)$.

The Jeans equation (Property 6.6) becomes a total derivative after multiplying by
$g(r) = \exp\int 2\beta\,\mathrm dr/r$, so that

$$ \nu_\star\sigma_r^2(r) = \frac{1}{g(r)} \int_r^\infty g(s)\,\nu_\star(s)\,\frac{GM(s)}{s^2}\,\mathrm ds , $$

with $g = 1$ for $\beta = 0$ and $g = r^2 + r_a^2$ for Osipkov-Merritt anisotropy. The projection
(Property 6.9) is done along the line of sight, $r = \sqrt{R^2 + z^2}$:

$$ \sigma_{\rm los}^2(R) = \frac{\int_0^\infty (1 - \beta R^2/r^2)\,\nu_\star\sigma_r^2\,\mathrm dz}
{\int_0^\infty \nu_\star\,\mathrm dz} . $$

The model functions and the projection are given. **TODO B.1** - implement `nu_sigr2`.

In [ ]:
G = 1.0


def nu_tracer(r):
    # Plummer tracer, scale radius 1, arbitrary normalisation.
    return (1.0 + r**2) ** -2.5


def halo_mass(r, v0, rc):
    # Cored halo, v_c^2 = v0^2 r^2 / (r^2 + rc^2).
    return v0**2 * r**3 / (G * (r**2 + rc**2))


# Anisotropy profiles and the logarithm of the integrating factor g(r).
beta_iso = lambda r: 0.0 * r
logg_iso = lambda r: 0.0 * r


def om_profiles(ra):
    beta = lambda r: r**2 / (r**2 + ra**2)
    logg = lambda r: np.log(r**2 + ra**2)
    return beta, logg


def nu_sigr2(rgrid, mass, logg):
    # nu sigma_r^2 on the radii rgrid.
    #   mass : callable M(r);  logg : callable ln g(r)
    # TODO B.1: for each r, integrate exp(logg(s) - logg(r)) * nu_tracer(s) * G * mass(s) / s**2
    # from r to infinity with quad.
    raise NotImplementedError("TODO B.1")


def sigma_los(R, mass, beta, logg, rmin=1e-3, rmax=1e3):
    # Line-of-sight dispersion at projected radii R (Property 6.9).
    rg = np.logspace(np.log10(rmin), np.log10(rmax), 300)
    spline = CubicSpline(np.log(rg), np.log(nu_sigr2(rg, mass, logg)))
    p = lambda r: np.exp(spline(np.log(np.clip(r, rmin, rmax))))
    out = np.empty_like(R)
    for k, Rk in enumerate(R):
        num = quad(lambda z: (1 - beta(np.hypot(Rk, z)) * Rk**2 / (Rk**2 + z**2))
                   * p(np.hypot(Rk, z)), 0, np.inf, limit=200)[0]
        den = quad(lambda z: nu_tracer(np.hypot(Rk, z)), 0, np.inf, limit=200)[0]
        out[k] = num / den
    return np.sqrt(out)

Test: the self-consistent isotropic Plummer sphere ($M = 1$, the tracer generates the potential)
must give $\sigma_{\rm los}^2 = 3\pi/(64\sqrt{1 + R^2})$ (Exercise 6.7).

In [ ]:
plummer_mass = lambda r: r**3 / (1 + r**2) ** 1.5
R_test = np.array([0.1, 1.0, 3.0])
print(sigma_los(R_test, plummer_mass, beta_iso, logg_iso) ** 2 / (3 * np.pi / (64 * np.sqrt(1 + R_test**2))))

**TODO B.2** - Model A is isotropic, with $v_0 = r_{\rm c} = 1$. Show analytically that
$\sigma_r^2 = v_0^2/5$ at every radius, then compute and plot $\sigma_{\rm los}(R)$ for
$0.02 \le R \le 6$.

In [ ]:
R = np.logspace(np.log10(0.02), np.log10(6.0), 40)
mass_A = lambda r: halo_mass(r, 1.0, 1.0)
# TODO B.2: sA = sigma_los(R, mass_A, beta_iso, logg_iso); plot it with a horizontal line at 1/sqrt(5).

**TODO B.3** - Model B has Osipkov-Merritt anisotropy with $r_a = 1$. Fit $(v_0, r_{\rm c})$ so that its
$\sigma_{\rm los}$ matches model A on 14 radii log-spaced in $0.05 \le R \le 4$: minimise
$\sum (\sigma_{\rm B}/\sigma_{\rm A} - 1)^2$ with `minimize(..., method="Nelder-Mead")`, fitting the
logarithms of the parameters. Plot both dispersion profiles and both circular-speed curves. Compare the
enclosed masses at $r = 0.1$, $1$ and $10$. Where are they equal?

In [ ]:
beta_B, logg_B = om_profiles(1.0)
R_fit = np.logspace(np.log10(0.05), np.log10(4.0), 14)


def loss(log_params):
    v0, rc = np.exp(log_params)
    # TODO B.3: return the sum of squared relative differences with model A on R_fit.
    raise NotImplementedError("TODO B.3")


# res = minimize(loss, np.log([1.5, 3.0]), method="Nelder-Mead")
# v0_B, rc_B = np.exp(res.x)

**TODO B.4** - Compute $\sigma_r$ and $\sigma_t = \sigma_r\sqrt{1-\beta}$ for both models at $r = 2$. How
precisely would proper-motion dispersions have to be measured to tell the two models apart?

In [ ]:
# TODO B.4: use nu_sigr2(np.array([2.0]), ...) / nu_tracer(2.0) for each model.